# オープンな LLM とチャットする (チャットテンプレート)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_llm/llm_chat_local.ipynb)

NLP クックブック 17-01 ／ 大規模言語モデル (LLM)

Hugging Face Hub の指示チューニング済みモデルを手元で動かして、システムプロンプト付きの対話をしたい。

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"   # 0.5B の小型日本語モデル。CPU でも動く
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                                             device_map="auto" if torch.cuda.is_available() else None).eval()

def chat(messages, max_new_tokens=200, **kwargs):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

messages = [
    {"role": "system", "content": "あなたは親切な日本語教師です。簡潔に答えてください。"},
    {"role": "user", "content": "「は」と「が」の違いを、例文を 1 つずつ挙げて説明してください。"},
]
reply = chat(messages)
print(reply)

「は」と「が」はどちらも接続助詞ですが、主に文の焦点や目的を示すために使用されます。

「は」は、文中の名詞や動詞の前後に置かれて、その名詞や動詞が何かについて言及していることを示します。例えば、「この花は美しいです」という文では、「は」が花を指しています。この場合、花は美しさの焦点となっています。

一方、「が」は、動作や状態の対象を示すために使用されます。例えば、「私が勉強しています」という文では、「が」が勉強している対象である私を指し示しています。この場合、「が」は動作の結果や原因を表すために用いられます。

このように、「は」は焦点を示し、「が」は対象を示すという違いがあります。これにより、文の意味やニュアンスが明確に伝わります。


In [2]:
# 対話を続ける: これまでのやりとりをすべて messages に積んで渡す (モデル自体は状態を持たない)
messages += [{"role": "assistant", "content": reply}, {"role": "user", "content": "今の説明を、小学生にもわかるように一文で言い直してください。"}]
print(chat(messages))

「は」は focus（焦点）を示し、「が」は thing（対象）を示す接続助詞です。例えば、「は」は花について話していて、「が」は勉強している相手を指し示しています。


In [3]:
# チャットテンプレートが実際に作る文字列を確認する
print(tokenizer.apply_chat_template(messages[:2], add_generation_prompt=True, tokenize=False))

<|system|>あなたは親切な日本語教師です。簡潔に答えてください。</s><|user|>「は」と「が」の違いを、例文を 1 つずつ挙げて説明してください。</s><|assistant|>


- 0.5B のモデルは日本語としては流暢だが、内容は不正確なことが多い (上の説明も文法的には誤りを含む)。動作確認用と割り切り、実用には大きなモデルを使う。
- モデルごとに特殊トークンの形式が違う。自分で文字列を組み立てず、必ず `apply_chat_template` を使う。
- `add_generation_prompt=True` で「ここからアシスタントの発話」という印が末尾に付く。忘れるとモデルがユーザーの発話の続きを書いてしまう。
- 同じコードで、GPU があれば `Qwen/Qwen2.5-7B-Instruct`、`google/gemma-3-4b-it`、`sbintuitions/sarashina2.2-3b-instruct-v0.1` などのより大きなモデルに差し替えられる。